In [57]:
import json

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------

with open("experimental_outputs/label_change_intervention_results.json", "r") as f:
    data = json.load(f)

VALID_SUBSETS = {"true", "false"}
VALID_STRENGTHS = {-2, -1, 0, 1, 2}
VALID_LAYERS = {9, 10, 11, 12, 13}

# --------------------------------------------------
# FILTER + BUILD DICTIONARY
# --------------------------------------------------

result = {}

for d in data:
    # filter subset
    if d.get("subset") not in VALID_SUBSETS:
        continue

    # filter intervention strength
    strength = d.get("intervention_strength")
    if strength not in VALID_STRENGTHS:
        continue

    # filter hidden state layers
    layers = {hs[0] for hs in d.get("hidden_states", [])}
    if not layers.issubset(VALID_LAYERS):
        continue

    model = d["model"]
    probe = d["probe class"]
    subset = d["subset"]
    p_diff = d["p_diff"]

    # train_datasets is a list → make it hashable & order-invariant
    train_dataset = "+".join(sorted(d["train_datasets"]))

    key = (model, probe, train_dataset, subset, strength)
    result[key] = p_diff

# --------------------------------------------------
# RESULT
# --------------------------------------------------

print(f"Number of entries: {len(result)}")


Number of entries: 134


In [58]:
model = 'llama-3.2-3B-Instruct'
probe = 'MMProbe'
train_dataset = 'random'

strength = 2
NIE_false_to_true = (result[(model, probe, train_dataset, 'false', strength)] - result[(model, probe, train_dataset, 'false', 0)]) / (result[(model, probe, train_dataset, 'true', 0)] - result[(model, probe, train_dataset, 'false', 0)])
NIE_true_to_false = (result[(model, probe, train_dataset, 'true', -strength)] - result[(model, probe, train_dataset, 'true', 0)]) / (result[(model, probe, train_dataset, 'false', 0)] - result[(model, probe, train_dataset, 'true', 0)])
print(NIE_false_to_true, NIE_true_to_false)

0.0 -0.004338394793926247


In [59]:
import pandas as pd
from collections import defaultdict
strength = 2  # you can loop over this later if needed
rows = []

# get unique dimensions from result keys
models = sorted({k[0] for k in result})
probes = sorted({k[1] for k in result})
train_datasets = sorted({k[2] for k in result})

for probe in probes:
    for train_dataset in train_datasets:
        row = {
            "probe": probe,
            "train_dataset": train_dataset,
        }

        for model in models:
            try:
                # baseline terms
                f0 = result[(model, probe, train_dataset, "false", 0)]
                t0 = result[(model, probe, train_dataset, "true", 0)]

                # strength terms
                f_pos = result[(model, probe, train_dataset, "false", strength)]
                t_neg = result[(model, probe, train_dataset, "true", -strength)]

                NIE_false_to_true = (f_pos - f0) / (t0 - f0)
                NIE_true_to_false = (t_neg - t0) / (f0 - t0)

                row[(model, "false→true")] = NIE_false_to_true
                row[(model, "true→false")] = NIE_true_to_false

            except KeyError:
                # missing data → leave as NaN
                row[(model, "false→true")] = float("nan")
                row[(model, "true→false")] = float("nan")

        rows.append(row)
df = pd.DataFrame(rows)
df = df.set_index(["probe", "train_dataset"])
df

(llama-3.2-3B, false→true)  \
probe   train_dataset                                          
LRProbe cities                                      0.005882   
        cities+neg_cities                           0.029412   
        larger_than                                 0.014706   
        larger_than+smaller_than                    0.005882   
        likely                                     -0.002941   
        random                                           NaN   
MMProbe cities                                      0.100629   
        cities+neg_cities                           0.122642   
        larger_than                                 0.044025   
        larger_than+smaller_than                    0.015723   
        likely                                      0.072327   
        random                                      0.012579   

                                  (llama-3.2-3B, true→false)  \
probe   train_dataset                                          
LRProbe cities                                      0.002941   
        cities+neg_cities                           0.032353   
        larger_than                                 0.014706   
        larger_than+smaller_than                    0.005882   
        likely                                      0.002941   
        random                                           NaN   
MMProbe cities                                      0.122642   
        cities+neg_cities                           0.165094   
        larger_than                                 0.066038   
        larger_than+smaller_than                    0.025157   
        likely                                      0.059748   
        random                                     -0.000000   

                                  (llama-3.2-3B-Instruct, false→true)  \
probe   train_dataset                                                   
LRProbe cities                                               0.013015   
        cities+neg_cities                                    0.013015   
        larger_than                                          0.006508   
        larger_than+smaller_than                             0.002169   
        likely                                               0.000000   
        random                                                    NaN   
MMProbe cities                                               0.071584   
        cities+neg_cities                                    0.069414   
        larger_than                                          0.030369   
        larger_than+smaller_than                             0.010846   
        likely                                               0.010846   
        random                                               0.000000   

                                  (llama-3.2-3B-Instruct, true→false)  
probe   train_dataset                                                  
LRProbe cities                                               0.084599  
        cities+neg_cities                                    0.082430  
        larger_than                                          0.034707  
        larger_than+smaller_than                             0.021692  
        likely                                              -0.000000  
        random                                                    NaN  
MMProbe cities                                               0.859002  
        cities+neg_cities                                    0.654013  
        larger_than                                          0.267896  
        larger_than+smaller_than                             0.086768  
        likely                                               0.056399  
        random                                              -0.004338